<a href="https://colab.research.google.com/github/asilbek1301/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **JaydariGPT | Fine-tuning a small AI model**
* **Goal:** teach TinyLlama, a small pretrained chat model, to follow instructions
* **Method:** QLoRA. Shrink the model to 4-bit, freeze it, then train a tiny add-on called a LoRA adapter
* **Training data:** 7,000 examples from the Alpaca instruction dataset
* **Result:** a `jaydari_gpt` folder with the adapter and tokenizer files. TinyLlama itself is not saved there, so it is downloaded again when the adapter is used
* **Runs on:** Google Colab with a GPU. The saved outputs below came from a T4 GPU

```text
TinyLlama (1.1B parameters)
    |  quantization: shrink to 4-bit
    v
Frozen 4-bit model
    |  PEFT: add small LoRA layers
    v
Model + LoRA (only 0.1% is trainable)
    |  SFTTrainer: 2 passes over 7,000 examples
    v
LoRA adapter saved to jaydari_gpt/
    |  PeftModel: load TinyLlama + adapter
    v
ask_jaydari_gpt("What is Neural Network?")
```

**About this notebook**
* All code comes from `jaydari_gpt.ipynb` unchanged, including every comment and commented-out line
* Long cells are split into smaller ones, and the zip and download cells now sit right after training
* The outputs are the ones saved in `jaydari_gpt.ipynb`. They were not re-run here
* **Watch out** marks a place where the code or its saved output has a problem

# **Setup | install libraries**
* `transformers`: downloads and runs pretrained AI models
* `torch`: PyTorch, the engine that does the math on the GPU
* `bitsandbytes`: shrinks the model to 4-bit
* `datasets`: downloads and prepares training data
* `peft`: adds the small trainable LoRA layers
* `trl`: provides `SFTTrainer`, the training loop

In [47]:
!pip install transformers torch bitsandbytes datasets peft trl
!pip uninstall -y torchao

# **Setup | import tools**
* `AutoTokenizer`: turns text into numbers and back
* `BitsAndBytesConfig`: the settings for shrinking the model
* `AutoModelForCausalLM`: loads a text-generating model
* `TrainingArguments`: the settings for training
* `load_dataset`: downloads a dataset
* `LoraConfig`, `get_peft_model`: define the LoRA layers and attach them
* `SFTTrainer`: runs the training

In [48]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# **Tokenizer | AutoTokenizer**
* `model_id` is the name of the base model on Hugging Face
* The tokenizer converts text to token numbers before the model reads it, and back to text afterwards
* It must come from the same model, because each model has its own vocabulary
* The two commented-out `print` lines show the vocabulary size and special tokens
* More detail on tokenizers is in `tekoneization.ipynb`

In [3]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' # Base model | pretrained model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# **Quantization | BitsAndBytesConfig**
* Quantization stores the weights with less precision, so the model takes much less GPU memory
* The comment's comparison explains it well: like turning a 4K video into 1080p, it gets smaller and loses a little quality

| Setting | Value | Meaning |
|---|---|---|
| `load_in_4bit` | `True` | Store each weight in 4 bits |
| `bnb_4bit_use_double_quant` | `True` | Compress the quantization data a second time to save a bit more memory |
| `bnb_4bit_compute_dtype` | `torch.bfloat16` | Weights are stored in 4 bits, but calculations run in bfloat16 |
| `bnb_4bit_quant_type` | `'nf4'` | The 4-bit method, built for the kind of values found in AI models |

In [4]:
# quantization    => original modelni hajmini kichraytirib config qilib yuklash
# 4K => 1080p
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

# **Load the model | AutoModelForCausalLM**
* Downloads TinyLlama and applies the 4-bit settings while loading
* "Causal LM" means a model that writes text by predicting the next token, again and again
* `device_map='auto'` puts the model on the GPU, or on the CPU if there is no GPU
* The commented-out `dtype` line is an option that is not used here

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    # dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# **Before Fine-tuning | first test**
* Asks the model a question before any training, so there is something to compare against later
* `tokenizer(...)` turns the prompt into numbers, and `.to(model.device)` moves them to the same place as the model
* `torch.no_grad()` turns off training math, which saves memory during generation
* `max_new_tokens=80`: write at most 80 new tokens
* `do_sample=True` with `temperature=0.7`: pick words with some randomness. Lower temperature gives safer answers
* `tokenizer.decode(...)` turns the result back into text

**Reading the output:** only the prompt is printed, so the model gave no answer. TinyLlama-Chat expects its own chat format, so a bare sentence probably makes it stop straight away.

In [6]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is?"
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? How does it break down words into smaller pieces? Can you provide an example of how a tokenizer works?


# **Model architecture | inside TinyLlama**
* TinyLlama is a stack of 22 identical blocks. `model.model.layers[0]` is the first one
* `Linear4bit` in the output confirms the layers really are quantized
* The commented-out `print(model)` line would print the whole model

**Inside one block**

| Part | Job |
|---|---|
| `self_attn` | Attention: decides which earlier words matter for the next word |
| `q_proj` | Query: what this word is looking for |
| `k_proj` | Key: what each word offers |
| `v_proj` | Value: the information each word passes on |
| `o_proj` | Output: combines the attention result |
| `mlp` | Processes each word further after attention |
| `input_layernorm`, `post_attention_layernorm` | Keep the numbers in a stable range |

**Key numbers from `model.config`**
* `vocab_size`: 32,000 different tokens
* `hidden_size`: each token is a list of 2,048 numbers
* `num_hidden_layers`: 22 blocks
* `num_attention_heads`: 32
* `max_position_embeddings`: 2,048 tokens of text at most

In [7]:
# print(model)
first_block = model.model.layers[0]
print('first_block:', first_block)
print('==============')
print(first_block.self_attn)
print('==============')
print(model.config)

first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (o_pr

# **Model size | count_parameters**
* A parameter is one number the model learned. `numel()` counts them for each layer, and `sum` adds them up
* The printed total is 615,606,272, which is lower than the 1.1B in the model's name
* The reason is that in 4-bit layers two weights share one byte, so `numel()` shows those layers at half their real size
* The real total is 1,100,048,384

| Part | Real count | Counted here |
|---|---:|---:|
| Linear layers in the 22 blocks (4-bit) | 968,884,224 | 484,442,112 |
| Embeddings, `lm_head`, norms (not 4-bit) | 131,164,160 | 131,164,160 |
| **Total** | **1,100,048,384** | **615,606,272** |

In [8]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


# **datasets library | load_dataset**
* instruction tuning
* Instruction tuning means training on pairs of "here is a task" and "here is a good answer"
* `yahma/alpaca-cleaned` has 51,760 of these examples
* `split="train"` selects the training part of the dataset
* Each row has 3 columns: `instruction` (the task), `input` (extra context, often empty) and `output` (the answer)
* Only the last line of the cell is displayed, so `dataset[1]` appears and `dataset` does not
* The `HF_TOKEN` warning is harmless. Downloads without a Hugging Face login are just slower

In [9]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

# **Prompt format | generate_prompt**
* The model reads one continuous text, so the 3 columns are joined into one
* `if input_text:` picks the template with an `### Input:` section, and `else` picks the one without it
* `### Instruction:` and `### Response:` teach the model where the task ends and the answer starts
* `formatting_func` returns the text as a new `text` column
* `dataset.map(...)` runs that on all 51,760 rows

**With input**
```text
### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
```

**Without input**
```text
### Instruction:
{instruction}

### Response:
{output}
```

In [10]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}"
      )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

# **Prompt format | check the result**
* Shows the new `text` column for the first row
* `\n` means a new line

In [11]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

# **Dataset size | select**
* Keeps 7,000 of the 51,760 rows, about 13.5%, so training finishes in a reasonable time
* `range(7000)` takes the first 7,000 rows in file order

**Watch out:** `select` runs before `shuffle`, so the other 44,760 rows can never be used. To sample from the whole dataset, shuffle first and then select.

In [12]:
dataset = dataset.select(range(7000))

# **Shuffle | seed**
* Mixes the row order so the model does not see examples in the same order as the file
* `seed=42` makes the random mix repeatable
* The first note below shows this: two runs with the same seed give the exact same order
* The second note compares two lists of example numbers
* `dataset` at the end confirms the result: 7,000 rows and 4 columns

In [13]:
dataset = dataset.shuffle(seed=42)

In [14]:
# [7, 2, 4, 1, 5, 3, 9, 0, 3, 4, 5, 0]
# [7, 2, 4, 1, 5, 3, 9, 0, 3, 4, 5, 0] EXACT SAME ORDER <-- this is what seed stands for

In [15]:
# [7, 2, 4, 1, 5, 3, 9, 0, 3, 4, 5, 0]
# [337, 142, 5424, 5321, 7545, 853, 69, 2430, 243, 124, 457, 50]

In [16]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

# **Full Fine-tuning  =>  Cheap Fine-tuning**
* **Full fine-tuning** updates all 1.1B parameters. On a small GPU this runs out of memory, which is the OOM in the note below
* **Cheap fine-tuning** freezes those parameters and trains only a small new part
* PEFT is the library that does this
* The Uzbek note says: freeze all 22 hidden layers of the pretrained model, and add a small trainable layer with PEFT

In [17]:
# PEFT => Parameter Efficient Fine Tuning
# pretrained ai modelning barcha 22 hidden layerlarini muzlatib olamiz va PEFT orqali kichik, trainable layer qo'shib olamiz

# OOM => Out of Memory

# **LoRA | LoraConfig**
* LoRA adds two very small layers beside an existing layer. Training changes only these small layers

| Setting | Value | Meaning |
|---|---|---|
| `r` | `8` | Rank: the width of the small layers. A bigger value adds more parameters |
| `lora_alpha` | `16` | Scaling: how strongly LoRA affects the result. The strength is `lora_alpha / r`, so 2 here |
| `lora_dropout` | `0.05` | Randomly ignores 5% of values during training, so the model memorizes less |
| `task_type` | `"CAUSAL_LM"` | Tells PEFT this is a text-generating model |
| `target_modules` | `["q_proj", "v_proj"]` | Adds LoRA to the Query and Value layers in every block |

In [18]:
lora_config = LoraConfig(
    r=8,                                    # rank -- katta bolgani sari ko'p parametrlar qo'shiladi
    lora_alpha=16,                          # scaling
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# **Attach LoRA | get_peft_model**
* Freezes the original model and adds the LoRA layers from `lora_config`
* Each target layer gets two parts: `lora_A` shrinks the data down to 8 numbers, and `lora_B` expands it back
* The numbers below explain the printed result exactly

In [19]:
model = get_peft_model(model, lora_config)

# **Trainable parameters | print_trainable_parameters**
* Only 0.1023% of the model will be trained
* `all params` here is the real size: 1,100,048,384 from TinyLlama plus 1,126,400 from LoRA

| Layer | `lora_A` | `lora_B` | Total |
|---|---:|---:|---:|
| `q_proj` | 2048 × 8 = 16,384 | 8 × 2048 = 16,384 | 32,768 |
| `v_proj` | 2048 × 8 = 16,384 | 8 × 256 = 2,048 | 18,432 |
| **One block** | | | **51,200** |
| **All 22 blocks** | | | **1,126,400** |

In [20]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


# **LoRA vs QLoRA | two names**
* **LoRA** (Low-Rank Adaptation) is the method of training small add-on layers
* **QLoRA** is LoRA on a quantized base model
* This notebook uses QLoRA, because the base model was loaded in 4-bit

In [21]:
# QLoRA
# LoRA

# **Training settings | TrainingArguments**

| Setting | Value | Meaning |
|---|---|---|
| `per_device_train_batch_size` | `1` | One example at a time, which suits a small GPU |
| `gradient_accumulation_steps` | `4` | Update the weights once every 4 examples. This learns like a batch of 4 with the memory of a batch of 1 |
| `num_train_epochs` | `2` | Go through the dataset 2 times. The `overfit` comment is a reminder that too many passes make the model memorize |
| `logging_steps` | `20` | Report progress every 20 weight updates |
| `output_dir` | `"./jaydari_gpt"` | Folder for checkpoints |
| `save_strategy` | `"epoch"` | Save a checkpoint after each pass |
| `bf16` | `True` | Train in 16-bit bfloat16 to use less memory |
| `fp16` | `False` | Do not use the other 16-bit format |
| `report_to` | `"none"` | Turn off extra logging services |

**How many updates?**
* 7,000 examples ÷ 4 = 1,750 updates per pass
* 1,750 × 2 passes = 3,500 updates
* That is why the saved checkpoints are named `checkpoint-1750` and `checkpoint-3500`

**Watch out:** the comment on `logging_steps` says every 10, but the value is 20.

In [27]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,      # batch size -- kichik gpu lar bilan llm training uchun qulay
    gradient_accumulation_steps=4,      # VRAM ni kamroq ishlatishga yordam beradi
    num_train_epochs=2,                 # overfit
    logging_steps=20,                   # har 10ta ma'lumotni natijalarni ko'rsatib turadi
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    bf16=True,
    fp16=False,                         # xotira kamroq ishlatishda yordam beradi
    report_to="none"                    # qo'shimcha loglarni o'chirib beradi
)

# **Training | SFTTrainer**
* SFT means Supervised Fine-Tuning: learning from examples that include the correct answer
* The first cell checks that the `text` column exists, because that is what the trainer reads
* `formatting_func=lambda x: x['text']` tells the trainer to use that column
* `trainer.train()` does the training
* `model.save_pretrained(...)` saves only the small LoRA adapter, because `model` is a PEFT model
* `tokenizer.save_pretrained(...)` saves the tokenizer beside it

**Watch out: this cell crashed in its last saved run, as the error below shows.**
* With `bf16=True`, training hit a bfloat16 operation that needs a GPU with compute capability 8.0 or higher. The T4 has 7.5
* The crash happened at `trainer.train()`, so this run saved nothing
* The `jaydari_gpt.zip` tested later contains `checkpoint-3500`, so it must come from an earlier run that finished
* No training loss is recorded in this notebook

**Two possible fixes, neither tested yet:**
1. Choose a newer Colab GPU, such as L4 or A100
2. Stay on T4: set `bf16=False`, `fp16=True` and `bnb_4bit_compute_dtype=torch.float16`, then re-run from the model loading cell

In [28]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [29]:
# SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
)

trainer.train()
model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')

Step,Training Loss
20,1.522227
40,1.575482
60,1.314538
80,1.366647
100,1.303763
120,1.324687
140,1.350165
160,1.245642
180,1.309803
200,1.217429


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

# **Save your work | Google Drive or zip**
* The `jaydari_gpt` folder sits on Colab's temporary disk and is lost when the runtime ends
* **Option 1, Google Drive:** the commented-out cell mounts Drive and copies the folder there
* **Option 2, download:** `zip -r` packs the folder into `jaydari_gpt.zip`, then `files.download` sends it to the computer

**Watch out:** `files.download("jaydari_gpt")` points at the folder, but it needs a single file. The zip file is `"jaydari_gpt.zip"`.

In [31]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp -r jaydari_gpt /content/drive/MyDrive/

In [30]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1750/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-1750/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/optimizer.pt (deflated 22%)
  adding: jaydari_gpt/checkpoint-1750/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-1750/adapter_model.safetensors (deflated 23%)
  adding: jaydari_gpt/checkpoint-1750/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-1750/trainer_state.json (deflated 80%)
  adding: jaydari_gpt/checkpoint-1750/scheduler.pt (deflated 61%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-1750/adapter_config.json (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/training_args.bin (deflated 53%)
  adding: jaydari_gpt/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/adapter_mo

In [37]:
from google.colab import files
files.download("jaydari_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Base model | clean copy for comparison**
* Loads TinyLlama again with no LoRA and no 4-bit
* The output shows plain `Linear` layers instead of `Linear4bit`
* `model_base.eval()` switches the model from training mode to answering mode
* The names end in `_base` so they do not replace `model` and `tokenizer`

In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

# **Base model | test**
* Same steps as the first test, with `max_new_tokens=120`

**Reading the output:** again only the prompt is printed, so the base model gave no answer to this plain prompt.

In [39]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words.


# **FINE-TUNED MODEL INFERENCE**
* Inference means using a trained model to get answers
* `model_path = "jaydari_gpt"` points at the saved adapter folder
* `AutoModelForCausalLM` finds the adapter there, then loads TinyLlama and adds the adapter
* That is why there are two progress bars. 88 is the number of LoRA weights: 22 blocks × 2 layers × 2 parts
* This replaces the earlier `model` and `tokenizer`

In [49]:
model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

# **Fine-tuned model | test**
* The prompt uses the `### Instruction:` and `### Response:` format from training
* The Uzbek instruction means "Give information about machine learning."

**Watch out:**
* The answer is wrong: it calls the text a common phrase in Azeri. The training data is English, so the model has not learned Uzbek
* This is not a fair comparison with the base model, because the language and the format differ. To compare them, ask both the same question

In [50]:
prompt = """### Instruction:
Mashinaviy o'qitish haqida ma'lumot ber.

### Response:
"""


inputs_ft = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ft = model.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Mashinaviy o'qitish haqida ma'lumot ber.

### Response:
The following is a translation of the text:

"What a great book! It is really interesting and informative."


# **MODEL TESTING | start from the saved zip**
* This part works on its own, without training again
* It needs `jaydari_gpt.zip` uploaded to Colab. On a brand new runtime, run the install cell at the top first
* `langdetect` guesses the language of a piece of text

In [51]:
!pip install -q langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **Unzip | restore the adapter**
* `!unzip` restores the `jaydari_gpt` folder
* Main files: `adapter_model.safetensors` (the trained LoRA weights) and `adapter_config.json` (its settings)
* `checkpoint-1750` and `checkpoint-3500` are the saves from the end of each pass

**Watch out:** the zip also contains `GitHub.app.zip`, which has nothing to do with the model. It looks like an accidental extra file.

In [53]:
!unzip jaydari_gpt.zip

Archive:  jaydari_gpt.zip
   creating: jaydari_gpt/
   creating: jaydari_gpt/checkpoint-1750/
  inflating: jaydari_gpt/checkpoint-1750/tokenizer.json  
  inflating: jaydari_gpt/checkpoint-1750/chat_template.jinja  
  inflating: jaydari_gpt/checkpoint-1750/optimizer.pt  
  inflating: jaydari_gpt/checkpoint-1750/rng_state.pth  
  inflating: jaydari_gpt/checkpoint-1750/adapter_model.safetensors  
  inflating: jaydari_gpt/checkpoint-1750/README.md  
  inflating: jaydari_gpt/checkpoint-1750/trainer_state.json  
  inflating: jaydari_gpt/checkpoint-1750/scheduler.pt  
  inflating: jaydari_gpt/checkpoint-1750/tokenizer_config.json  
  inflating: jaydari_gpt/checkpoint-1750/adapter_config.json  
  inflating: jaydari_gpt/checkpoint-1750/training_args.bin  
  inflating: jaydari_gpt/tokenizer.json  
  inflating: jaydari_gpt/chat_template.jinja  
  inflating: jaydari_gpt/adapter_model.safetensors  
  inflating: jaydari_gpt/README.md   
   creating: jaydari_gpt/checkpoint-3500/
  inflating: jaydari_

# **Load adapter | PeftModel**
1. Load the tokenizer from the adapter folder. `pad_token = eos_token` reuses the end token for padding
2. Load the original TinyLlama, this time in bfloat16 instead of 4-bit
3. `PeftModel.from_pretrained` puts the trained adapter on top
4. `model.eval()` switches to answering mode

* The output shows the result: `q_proj` and `v_proj` are now `lora.Linear` layers with `lora_A` and `lora_B`
* `/content/` is the Colab path. It will not exist on another computer
* `PeftConfig` is imported but not used

In [54]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

# **Chat function | ask_jaydari_gpt**
1. **Detect the language** of the question with `detect_langs`. If that fails, the language is `"unknown"` with confidence 0.0
2. **Block non-English questions**, because the model only trained on English. All 3 conditions must be true: not English, confidence of at least 0.95, and more than 20 characters
3. **Build the prompt** in the same format as training
4. **Generate the answer** with `temperature=0.7` and `top_p=0.9`. `top_p` limits choices to the most likely words, and `eos_token_id` lets the model stop when it finishes
5. **Remove the prompt** with `decoded[len(full_prompt):]`, so only the answer is returned

**Watch out:**
* `.to("cuda")` is written directly in the function, so it fails on a computer without a GPU
* Short non-English questions of 20 characters or fewer are not blocked

In [55]:
from langdetect import detect_langs

def ask_jaydari_gpt(prompt: str, max_new_tokens=256):
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        return "Sorry, you should ask a question only in English."

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

# **Final test | ask a question**
* The language check prints `en (1.00)`, so the question goes through
* The result is an on-topic answer in English, using the adapter from the earlier finished run

In [56]:
ask_jaydari_gpt('What is Neural Network?')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: en (1.00)


'Neural Network is a type of artificial neural system that can process and learn information from data. It is a network of nodes that can be represented as a graph, where each node can be connected to other nodes through a directed edge. The nodes in the network are called neurons, and they are connected to one another through synapses, which are the connections between them.\n\nA neural network can be used to perform a variety of tasks, such as pattern recognition, decision making, and language processing. The input data is passed through the network, and the neurons compute the output based on the information that they have received. The output is then used to make a decision or to generate a result.\n\nNeural networks can be trained to perform a particular task by providing it with a set of training data. The training data is used to map out the structure of the network, which is then used to predict the output based on the input. The learning process involves updating the weights a